# Adaptive Residual Steering (ARS) — Kaggle Execution Runner
## V17 — Phase 1 (V14 Legacy Pure) + Phase 2B Gate (Stronger Training)

### Diagnosis from V16:
- **Phase 1 works**: V14 config (+6.5pp) with legacy_pure, rho_max=0.05
- **Gate HURTS**: V16 SteerNet+Gate = +0.5pp vs SteerNet-only = +6.5pp
- **Gate barely trained**: 2 epochs, val_nll drop = 0.0012, weights at init
- **FINAL alpha = 0.38**: Gate suppresses final answer steering (should be 0.85)
- **FORMAT alpha = 0.40**: Gate doesn't suppress format enough (should be 0.15)
- **Root cause**: L1 penalty (0.01) + weak contrastive (0.5) → gate pushes everything down

### V17 Fixes:
1. **Phase 1**: Same V14 config — proven foundation
2. **Gate epochs**: 2 → **8** (let gate actually learn)
3. **Contrastive weight**: 0.5 → **2.0** (stronger math↑ format↓ signal)
4. **L1 weight**: 0.01 → **0.001** (stop pushing all alphas to zero)
5. **Dual benchmark**: Run BOTH fixed_no_gate AND fixed+gate to isolate gate effect

### Expected outcome:
- fixed_no_gate should match V14 (~+6.5pp) confirming Phase 1 is solid
- fixed+gate should improve over fixed_no_gate if gate learned properly
- Gate alpha: OPERATOR/RESULT ~0.85, FORMAT ~0.15, FINAL ~0.85

## 1. Environment & Hardware Check

In [ ]:
# Set CUDA allocator config BEFORE any torch/transformers import.
# expandable_segments:True lets the allocator satisfy fragmented-memory requests
# without requiring a single contiguous free block — critical for 10-hook training.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi
!pip install -q "transformers>=4.36.0" "datasets>=2.16.0" "accelerate>=0.25.0" matplotlib pandas scipy tqdm


## 2. Clone Repository (`development` branch)

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"
import sys

REPO_DIR = "/kaggle/working/adaptive-residual-steering"
BRANCH = "development"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Abdullah182155/adaptive-residual-steering.git {REPO_DIR}
else:
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Per-Layer Norm Calibration (V5)

In [ ]:
import os
os.chdir(REPO_DIR)
print(f"Current directory: {os.getcwd()}")

# V10: 4 layers matching old notebook [17, 20, 23, 26]
NEW_LAYERS = '17 20 23 26'
print(f"Calibrating norms for 4-layer route: [{NEW_LAYERS}]")
!python scripts/calibrate_norms.py \
    --n_samples 5000 \
    --output norms_calibration.json \
    --candidate_layers {NEW_LAYERS}

print("\n✅ Norm calibration complete:")
!ls -lh norms_calibration.json

## 4. Verify Codebase Integrity & Unit Tests

In [ ]:
import os
import sys
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Current directory: {os.getcwd()}")
print(f"Tests directory exists: {os.path.exists('tests/')}")
!pytest tests/ -q --tb=short

## 5. Architecture Configuration

In [ ]:
# v5: rank-64 with causal convolution and per-layer caps
# v4: legacy architecture (default, for NLL training)
STEERNET_ARCH_ARGS = "--steernet_arch v5"
print(f"Architecture: {STEERNET_ARCH_ARGS}")

## 5b. Clear Stale Cache


In [ ]:
# ========== PHASE 1A→1B BUG-6 FIX: Clear Stale Cache ==========
# Cache fingerprint was missing κ and architecture, causing stale output reuse.
# This is now fixed in benchmark.py, but we clear the cache to be safe.
import shutil
import os

cache_dir = "./cache/benchmark_outputs"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"✅ Cleared stale cache: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)
else:
    print(f"✅ Cache directory clean: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)

## 6. Hyperparameter Configuration — V20 Method Overhaul

**V18** (architecture): mag_head OFF, soft_cap OFF, init_gain=0.02, conservative gate init

**V19** (methods): gen-based gate reward, on-policy Phase 1

**V20** (methods):
- **Gate**: GRPO normalization (4 samples, normalized advantages → 2-3 epoch convergence)
- **Gate**: Binary entropy reg replaces L1 (crisp 0/1 decisions, works WITH contrastive)
- **Phase 1**: Answer region upweighting (2× weight on final 30% tokens, was ignored in legacy_pure)
- **Phase 1**: Checkpoint by result_top1 (better proxy for test accuracy than weighted NLL)

In [ ]:
# ==============================================================================
# V22: PER-TOKEN MAGNITUDE + SUPERVISED GATE + KL/SHOT FIXES
# ==============================================================================
S1_OBJECTIVE = 'nll'
K_SCALE_MODE = 'sqrt'
BATCH_SIZE   = 2
GRAD_ACCUM   = 8
MAX_TOK_LEN  = 512

# -- Architecture: V5 mag_head enabled (V22 key change) --
AMPLITUDE_MODE = 'relative'       # V22: back to relative (not norm_capped)
RHO_MAX        = 0.10             # V22: raised from 0.08; mag_head starts at ~5%
STEER_SCALE    = 0.30
STEER_KAPPA    = 0.15
LORA_RANK      = 64
LORA_ALPHA     = 32.0
MAG_HEAD_INIT  = -3.0
STEERNET_INIT_GAIN    = 1.0       # V22: direction unit-normalized; init_gain = feature expressivity
STEERNET_USE_MAG_HEAD = True      # V22: per-token magnitude via sigmoid(mag_head(x))
STEERNET_USE_SOFT_CAP = False
STEERNET_USE_CONV     = True
PROMPT_SHIELDING      = False

# -- Loss: LEGACY PURE (V14 proven) --
LOSS_MODE           = 'legacy_pure'
TYPED_LEARNED_W     = False
LAMBDA_FORMAT       = 0.02
LAMBDA_PROSE        = 0.03
LAMBDA_OPERATOR     = 0.10
LAMBDA_OPERAND      = 0.15
LAMBDA_RESULT       = 0.40
LAMBDA_FINAL        = 0.30
FOCAL_GAMMA         = 2.0
KL_WEIGHT           = 0.0
KL_ARITH_WEIGHT     = 0.05        # V22: reduced from 0.30; allows arithmetic distribution shift
MAG_L1_WEIGHT       = 0.03        # V22: raised from 0.01; all token types now penalized

# -- Training --
# S1_LR = 1.5e-4 -- DO NOT CHANGE (V14 proven +6.5pp)
S1_LR          = 1.5e-4
S1_EPOCHS      = 5                # V22: extra epoch (gen-val still improving at epoch 4)
S1_TRAIN_N     = 7473
S1_VAL_N       = 600
S1_ANSWER_WEIGHT      = 2.0
S1_ANSWER_REGION_FRAC = 0.30
DIVERSIFY_PROMPTS     = True

# -- Shot distribution: 90% 0-shot focus (V22) --
TRAIN_SHOT_PROBS = '0.90,0.05,0.05,0.0,0.0'  # V22: was 0.50,0.10,0.20,0.05,0.15

# -- On-Policy Training (V22: frac=0.20, k=4 -> ~20min/epoch vs 3h) --
ON_POLICY              = True
ON_POLICY_START_EPOCH  = 1
ON_POLICY_K            = 4        # V22: reduced from 8 (saves 50% gen time)
ON_POLICY_FRAC         = 0.20     # V22: reduced from 0.50 (saves 60% gen time)
ON_POLICY_INTERVAL     = 1
ON_POLICY_GEN_BATCH    = 1        # OOM fix: 1*8=8 seq at once
ON_POLICY_GOLD_FALLBACK = False   # pure rejection sampling (no gold CoT)

# -- Generation / Benchmark --
BENCH_GEN_TEMP     = 0.0
BENCH_REP_PENALTY  = 1.1
BENCH_NO_REPEAT_NG = 0
BENCH_MAX_TOKENS   = 300

RUN_CONTRASTIVE = False
RUN_STAR        = False

CALIB_SEL_N  = 100
BENCH_N_TEST = 300
BENCH_SHOTS  = '0 2'

# -- Phase 2B Gate Configuration (V22: supervised advantage) --
GATE_EPOCHS              = 3
GATE_LR                  = 1e-4
GATE_FINETUNE_STEERNET   = True
GATE_STEERNET_LR         = 1e-5
GATE_INIT_ALPHA          = 0.25
GATE_USE_GLOBAL_CONTEXT  = False
GATE_CONTRASTIVE_WEIGHT  = 0.3    # V22: contrastive is auxiliary only
GATE_CONTRASTIVE_MODE    = 'hinge'
GATE_L1_WEIGHT           = 0.0
GATE_ENTROPY_WEIGHT      = 0.0    # V22: disabled (supervised mode)
GATE_RLOO_WEIGHT         = 0.0    # V22: disabled (supervised mode)
GATE_USE_TASK_NLL        = False
GATE_ANTISAT_WEIGHT      = 0.0    # V22: disabled
GATE_ANTISAT_THRESHOLD   = 0.85
GATE_RLOO_BASELINE       = 0.0
GATE_TRAINING_MODE       = 'supervised'   # V22: per-token advantage supervision
GATE_SUPERVISED_TEMP     = 5.0           # V22: sigmoid(advantage * temp) sharpness
GATE_GEN_REWARD_EVERY    = 0      # V22: disabled for supervised mode
GATE_GEN_SUBSET          = 4
GATE_GEN_WEIGHT          = 2.0
GATE_GEN_MAX_TOKENS      = 150

print(f'V22 | Phase 1: answer_w={S1_ANSWER_WEIGHT}x on final {S1_ANSWER_REGION_FRAC*100:.0f}% tokens')
print(f'SteerNet: gain={STEERNET_INIT_GAIN}, mag_head={STEERNET_USE_MAG_HEAD}, amplitude={AMPLITUDE_MODE}, rho_max={RHO_MAX}')
print(f'Shot probs: {TRAIN_SHOT_PROBS}  on_policy_frac={ON_POLICY_FRAC} k={ON_POLICY_K}')
print(f'Gate mode: {GATE_TRAINING_MODE}, adv_temp={GATE_SUPERVISED_TEMP}')
print(f'kl_arith={KL_ARITH_WEIGHT}  mag_l1={MAG_L1_WEIGHT}  epochs={S1_EPOCHS}')

In [ ]:
# ==============================================================================
# V18: Phase 2B Gate ENABLED — single 3-arm benchmark
# ==============================================================================
RUN_ROUTER = False
RUN_GATE   = True
BENCH_ARMS = 'baseline,fixed,gate'  # Single run: baseline + SteerNet-only + Gate+SteerNet

print(f'Router: DISABLED | Gate: ENABLED')
print(f'Benchmark: 3-arm single run — baseline, fixed (SteerNet-only), gate (SteerNet+Gate)')

## 7. Fixed Layer Configuration (Phase 1 V2)


In [ ]:
# ==============================================================================
# V10: 4 layers matching old notebook [17, 20, 23, 26]
# ==============================================================================
# Old notebook used these 4 mid-to-late layers with rank=64 each.
# Fewer layers = more per-layer amplitude, more focused steering.
FIXED_LAYERS = [17, 20, 23, 26]
S_FIXED = ','.join(map(str, FIXED_LAYERS))

print('='*80)
print(f'FIXED LAYER CONFIGURATION (V10 Classic — {len(FIXED_LAYERS)} Layers)')
print('='*80)
print(f'Active layers ({len(FIXED_LAYERS)}): {FIXED_LAYERS}')
print(f'Amplitude mode: {AMPLITUDE_MODE} (steer_scale={STEER_SCALE})')
print('='*80)

## 8. Stage S1: SteerNet Training (Legacy Pure NLL + Magnitude Regularization)

In [ ]:
# Stage S1: SteerNet Training — V22 + mag_head + 90% 0-shot + supervised gate
import subprocess

command = f'''python scripts/run_ars_pipeline.py --stage S1 {STEERNET_ARCH_ARGS}     --objective {S1_OBJECTIVE}     --s1_epochs {S1_EPOCHS}     --s1_lr {S1_LR}     --s1_train_n {S1_TRAIN_N}     --s1_val_n {S1_VAL_N}     --batch_size {BATCH_SIZE}     --steer_scale {STEER_SCALE}     --s_fixed {S_FIXED}     --set steer_kappa={STEER_KAPPA}     --set amplitude_mode={AMPLITUDE_MODE}     --set rho_max={RHO_MAX}     --set k_scale_mode={K_SCALE_MODE}     --set grad_accum={GRAD_ACCUM}     --set max_tok_len={MAX_TOK_LEN}     --set loss_mode={LOSS_MODE}     --set lora_rank={LORA_RANK}     --set lora_alpha={LORA_ALPHA}     --set mag_head_init_bias={MAG_HEAD_INIT}     --set diversify_prompts={str(DIVERSIFY_PROMPTS).lower()}     --set s1_answer_weight={S1_ANSWER_WEIGHT}     --set s1_answer_region_frac={S1_ANSWER_REGION_FRAC}     --set steernet_init_gain={STEERNET_INIT_GAIN}     --set steernet_use_mag_head={str(STEERNET_USE_MAG_HEAD).lower()}     --set steernet_use_soft_cap={str(STEERNET_USE_SOFT_CAP).lower()}     --set steernet_use_conv={str(STEERNET_USE_CONV).lower()}     --set prompt_shielding={str(PROMPT_SHIELDING).lower()}     --set mag_l1_weight={MAG_L1_WEIGHT}     --set kl_weight={KL_WEIGHT}     --set kl_arithmetic_weight={KL_ARITH_WEIGHT}     --set train_shot_probs={TRAIN_SHOT_PROBS}     --set generation_max_new_tokens={BENCH_MAX_TOKENS}     --set generation_repetition_penalty={BENCH_REP_PENALTY}     --set generation_no_repeat_ngram={BENCH_NO_REPEAT_NG}     --set on_policy={str(ON_POLICY).lower()}     --set on_policy_start_epoch={ON_POLICY_START_EPOCH}     --set on_policy_k={ON_POLICY_K}     --set on_policy_frac={ON_POLICY_FRAC}     --set on_policy_interval={ON_POLICY_INTERVAL}     --set on_policy_gen_batch={ON_POLICY_GEN_BATCH}     --set on_policy_gold_fallback={str(ON_POLICY_GOLD_FALLBACK).lower()}     --set candidate_layers={S_FIXED}     --set fixed_layers={S_FIXED}'''

process = subprocess.Popen(
    command, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")
if process.wait() != 0:
    raise RuntimeError(
        f"S1 pipeline failed with exit code {process.returncode}; "
        "full runner output is shown above."
    )

import os, glob
ckpts = sorted(glob.glob("checkpoints/steernet_epoch*.pt"))
print(f'
Per-epoch checkpoints saved: {[os.path.basename(c) for c in ckpts]}')
print('
'  + '='*80)
print('S1 Training Complete (V22 -- mag_head=True, relative, rho_max=0.10, 90pct 0-shot)')
print('='*80)

## 8b. Stage S2: Phase 2 Router Training (Per-Question Layer Selection)

In [ ]:
if RUN_ROUTER:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S2 {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n  {S1_VAL_N} \
        --s_fixed   {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --set steer_kappa={STEER_KAPPA} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set phase2_epochs={PHASE2_EPOCHS} \
        --set phase2_lr={PHASE2_LR} \
        --set router_max_k={ROUTER_MAX_K} \
        --set router_min_k={ROUTER_MIN_K} \
        --set router_warmup_epochs={ROUTER_WARMUP} \
        --set router_beta_entropy_subset={ROUTER_BETA_ENTROPY_SUBSET}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        print(f"[warning] S2 pipeline exited with code {process.returncode}")

    import torch, json
    ckpt_path = "checkpoints/router.pt"
    if os.path.exists(ckpt_path):
        print('\n' + '='*80)
        print('✅ S2 Router Training Complete')
        print('='*80)
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        print(f"Router checkpoint keys: {list(ckpt.keys())}")
        print(f"fixed_layers: {ckpt.get('fixed_layers')}")
        print(f"router_min_k: {ckpt.get('router_min_k')}  router_max_k: {ckpt.get('router_max_k')}")
else:
    print("Router training skipped (RUN_ROUTER=False)")

## 8c. Stage S2B: Phase 2B Gate Training (V18 Architecture Overhaul)

**V18 root-cause fixes** (from deep audit):
- **Conservative init**: gate_init_alpha=0.25 (sigmoid approx 0.25) — gate must EARN activation
- **Hinge contrastive**: ReLU one-sided, zero gradient once target met (was MSE)
- **No task NLL in gate loss**: task NLL overwhelmed contrastive, killed FINAL alpha
- **Anti-saturation**: penalizes alpha > threshold, per-layer tightening
- **RLOO baseline=0.0**: compare gate OFF vs learned (max contrast)
- **No global context**: removed 16 dead zero dims

In [ ]:
if RUN_GATE:
    import subprocess, torch, os

    command = (
        f"python scripts/run_ars_pipeline.py --stage S2B {STEERNET_ARCH_ARGS}"
        f" --s1_train_n {S1_TRAIN_N}"
        f" --s1_val_n {S1_VAL_N}"
        f" --s_fixed {S_FIXED}"
        f" --steer_scale {STEER_SCALE}"
        f" --batch_size {BATCH_SIZE}"
        f" --set steer_kappa={STEER_KAPPA}"
        f" --set amplitude_mode={AMPLITUDE_MODE}"
        f" --set rho_max={RHO_MAX}"
        f" --set k_scale_mode={K_SCALE_MODE}"
        f" --set lora_rank={LORA_RANK}"
        f" --set lora_alpha={LORA_ALPHA}"
        f" --set mag_head_init_bias={MAG_HEAD_INIT}"
        f" --set max_tok_len={MAX_TOK_LEN}"
        f" --set grad_accum={GRAD_ACCUM}"
        f" --set loss_mode={LOSS_MODE}"
        f" --set steernet_init_gain={STEERNET_INIT_GAIN}"
        f" --set steernet_use_mag_head={str(STEERNET_USE_MAG_HEAD).lower()}"
        f" --set steernet_use_soft_cap={str(STEERNET_USE_SOFT_CAP).lower()}"
        f" --set steernet_use_conv={str(STEERNET_USE_CONV).lower()}"
        f" --set prompt_shielding={str(PROMPT_SHIELDING).lower()}"
        f" --set use_gate=true"
        f" --set gate_epochs={GATE_EPOCHS}"
        f" --set gate_lr={GATE_LR}"
        f" --set gate_init_alpha={GATE_INIT_ALPHA}"
        f" --set gate_use_global_context={str(GATE_USE_GLOBAL_CONTEXT).lower()}"
        f" --set gate_finetune_steernet={str(GATE_FINETUNE_STEERNET).lower()}"
        f" --set gate_steernet_lr={GATE_STEERNET_LR}"
        f" --set gate_contrastive_weight={GATE_CONTRASTIVE_WEIGHT}"
        f" --set gate_contrastive_mode={GATE_CONTRASTIVE_MODE}"
        f" --set gate_l1_weight={GATE_L1_WEIGHT}"
        f" --set gate_entropy_weight={GATE_ENTROPY_WEIGHT}"
        f" --set gate_rloo_weight={GATE_RLOO_WEIGHT}"
        f" --set gate_use_task_nll={str(GATE_USE_TASK_NLL).lower()}"
        f" --set gate_antisat_weight={GATE_ANTISAT_WEIGHT}"
        f" --set gate_antisat_threshold={GATE_ANTISAT_THRESHOLD}"
        f" --set gate_rloo_baseline={GATE_RLOO_BASELINE}"
        f" --set gate_training_mode={GATE_TRAINING_MODE}"
        f" --set gate_supervised_temperature={GATE_SUPERVISED_TEMP}"
        f" --set gate_gen_reward_every={GATE_GEN_REWARD_EVERY}"
        f" --set gate_gen_subset={GATE_GEN_SUBSET}"
        f" --set gate_gen_weight={GATE_GEN_WEIGHT}"
        f" --set gate_gen_max_tokens={GATE_GEN_MAX_TOKENS}"
        f" --set candidate_layers={S_FIXED}"
        f" --set fixed_layers={S_FIXED}"
    )

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        print(f"[warning] S2B pipeline exited with code {process.returncode}")

    gate_path = "checkpoints/gate.pt"
    sn_gated_path = "checkpoints/steernet_gated.pt"
    if os.path.exists(gate_path):
        print("
" + "=" * 80)
        print("S2B Gate Training Complete (V22 -- supervised advantage)")
        print("=" * 80)
        ckpt = torch.load(gate_path, map_location="cpu", weights_only=False)
        improvement = ckpt.get("best_val_improvement", "N/A")
        if isinstance(improvement, float):
            print(f"Gate best val improvement (nll_base - nll_gate): {improvement:+.4f}")
        else:
            print(f"Gate best val improvement: {improvement}")
        print(f"Fixed layers: {ckpt.get('fixed_layers')}")
        print(f"Gate epochs: {ckpt.get('gate_epochs')}")
    if os.path.exists(sn_gated_path):
        print(f"Gated SteerNet checkpoint: {sn_gated_path}")
else:
    print("Gate training skipped (RUN_GATE=False)")

## 8d. Stage S1C: Contrastive Residual Training (Reasoning Direction)

**What it does:**
1. Generates greedy completions from the steered model for all training questions
2. Filters to questions where the model is WRONG (hard-example curriculum)
3. Computes the direction vector between correct and incorrect reasoning hidden states
4. Trains SteerNet delta to align with that direction (cosine similarity loss)

The contrastive loss teaches SteerNet WHERE to push in activation space for correct reasoning,
instead of just learning to mimic gold token sequences (which only learns format).

In [ ]:
if RUN_CONTRASTIVE:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S1C {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n {S1_VAL_N} \
        --batch_size {BATCH_SIZE} \
        --s_fixed {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --set steer_kappa={STEER_KAPPA} \
        --set rho_max={RHO_MAX} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set kl_weight={KL_WEIGHT} \
        --set loss_mode={LOSS_MODE} \
        --set grad_accum={GRAD_ACCUM} \
        --set max_tok_len={MAX_TOK_LEN} \
        --set contrastive_phase=true \
        --set contrastive_epochs={CONTRASTIVE_EPOCHS} \
        --set contrastive_lr={CONTRASTIVE_LR} \
        --set contrastive_weight={CONTRASTIVE_WEIGHT} \
        --set contrastive_nll_weight={CONTRASTIVE_NLL_W} \
        --set contrastive_frac={CONTRASTIVE_FRAC} \
        --set candidate_layers={S_FIXED} \
        --set fixed_layers={S_FIXED}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(
            f"S1C pipeline failed with exit code {process.returncode}; "
            "full runner output is shown above."
        )

    import os, glob
    ckpts = sorted(glob.glob("checkpoints/steernet_s1c_epoch*.pt"))
    print(f'\nS1C checkpoints: {[os.path.basename(c) for c in ckpts]}')
    print('\n' + '='*80)
    print('S1C Contrastive Training Complete')
    print('='*80)
else:
    print("Contrastive training skipped (RUN_CONTRASTIVE=False)")

## 8e. Stage S1R: Self-Taught Reasoning (STaR)

**What it does:**
1. Generates k=8 completions per question at temp=0.7 with steering ON
2. Scores each: keeps only **correct** completions (rejection sampling)
3. For hard questions (all 8 wrong): adds the gold answer as a **hint**, regenerates,
   then trains on the hint-generated solution WITHOUT the hint
4. Trains SteerNet with standard NLL on these self-generated correct solutions

**Why this works:** Training on gold CoT teaches FORMAT (the gold has specific wording).
Training on the model's OWN correct solutions teaches it to AMPLIFY reasoning paths
it already knows how to produce — much easier for SteerNet to learn.

In [ ]:
if RUN_STAR:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S1R {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n {S1_VAL_N} \
        --batch_size {BATCH_SIZE} \
        --s_fixed {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --set steer_kappa={STEER_KAPPA} \
        --set rho_max={RHO_MAX} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set kl_weight={KL_WEIGHT} \
        --set loss_mode={LOSS_MODE} \
        --set grad_accum={GRAD_ACCUM} \
        --set max_tok_len={MAX_TOK_LEN} \
        --set star_k={STAR_K} \
        --set star_temperature={STAR_TEMP} \
        --set star_epochs={STAR_EPOCHS} \
        --set star_lr={STAR_LR} \
        --set star_rounds={STAR_ROUNDS} \
        --set star_hint_k={STAR_HINT_K} \
        --set star_gen_batch={STAR_GEN_BATCH} \
        --set candidate_layers={S_FIXED} \
        --set fixed_layers={S_FIXED}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(
            f"S1R pipeline failed with exit code {process.returncode}; "
            "full runner output is shown above."
        )

    import os, glob
    ckpts = sorted(glob.glob("checkpoints/steernet_s1r_*.pt"))
    print(f'\nS1R checkpoints: {[os.path.basename(c) for c in ckpts]}')
    print('\n' + '='*80)
    print('S1R Self-Taught Reasoning Complete')
    print('='*80)
else:
    print("STaR training skipped (RUN_STAR=False)")

## 9. Stage S5: Deployment Calibration (Contract v3)


In [ ]:
# Stage S5: Calibration — writes deployment_contract.json (version 3)
!python scripts/run_ars_pipeline.py --stage S5 {STEERNET_ARCH_ARGS} \
    --calib_sel_n {CALIB_SEL_N} \
    --contract deployment_contract.json \
    --s_fixed {S_FIXED} \
    --steer_scale {STEER_SCALE} \
    --set steer_kappa={STEER_KAPPA} \
    --set amplitude_mode={AMPLITUDE_MODE} \
    --set rho_max={RHO_MAX} \
    --set k_scale_mode={K_SCALE_MODE} \
    --set lora_rank={LORA_RANK} \
    --set lora_alpha={LORA_ALPHA} \
    --set mag_head_init_bias={MAG_HEAD_INIT} \
    --set steernet_init_gain={STEERNET_INIT_GAIN} \
    --set steernet_use_mag_head={str(STEERNET_USE_MAG_HEAD).lower()} \
    --set steernet_use_soft_cap={str(STEERNET_USE_SOFT_CAP).lower()} \
    --set steernet_use_conv={str(STEERNET_USE_CONV).lower()} \
    --set prompt_shielding={str(PROMPT_SHIELDING).lower()} \
    --set candidate_layers={S_FIXED} \
    --set fixed_layers={S_FIXED}

import json as _json, pathlib as _pl
_c = _json.loads(_pl.Path("deployment_contract.json").read_text())
print(f"\nContract version: {_c.get('version')}  (expected 3)")
assert _c.get("version") == 3, "Expected contract v3!"
print("\n" + "="*80)
print("S5 Calibration Complete (Contract v3 verified)")
print("="*80)

## 10. Single 3-Arm Benchmark

V18: Single benchmark run with 3 arms:
1. **baseline** — Phi-2 without steering (evaluated ONCE)
2. **fixed** — SteerNet-only (Phase 1 weights, gate disabled)
3. **gate** — SteerNet + Gate joint (Phase 2B weights, gate active)


In [ ]:
import subprocess

_bench_cmd = f'''python scripts/run_ars_pipeline.py --stage bench {STEERNET_ARCH_ARGS} \
    --contract deployment_contract.json \
    --n_test {BENCH_N_TEST} \
    --shots {BENCH_SHOTS} \
    --report \
    --arms {BENCH_ARMS} \
    --record experiment_record.json \
    --s_fixed {S_FIXED} \
    --steer_scale {STEER_SCALE} \
    --set steer_kappa={STEER_KAPPA} \
    --set amplitude_mode={AMPLITUDE_MODE} \
    --set rho_max={RHO_MAX} \
    --set lora_rank={LORA_RANK} \
    --set lora_alpha={LORA_ALPHA} \
    --set mag_head_init_bias={MAG_HEAD_INIT} \
    --set steernet_init_gain={STEERNET_INIT_GAIN} \
    --set steernet_use_mag_head={str(STEERNET_USE_MAG_HEAD).lower()} \
    --set steernet_use_soft_cap={str(STEERNET_USE_SOFT_CAP).lower()} \
    --set steernet_use_conv={str(STEERNET_USE_CONV).lower()} \
    --set prompt_shielding={str(PROMPT_SHIELDING).lower()} \
    --set k_scale_mode={K_SCALE_MODE} \
    --set use_gate=true \
    --set gate_init_alpha={GATE_INIT_ALPHA} \
    --set gate_use_global_context={str(GATE_USE_GLOBAL_CONTEXT).lower()} \
    --set gate_contrastive_mode={GATE_CONTRASTIVE_MODE} \
    --set generation_temperature={BENCH_GEN_TEMP} \
    --set generation_repetition_penalty={BENCH_REP_PENALTY} \
    --set generation_no_repeat_ngram={BENCH_NO_REPEAT_NG} \
    --set generation_max_new_tokens={BENCH_MAX_TOKENS} \
    --set candidate_layers={S_FIXED} \
    --set fixed_layers={S_FIXED}'''

print("=" * 80)
print("  3-ARM BENCHMARK: baseline / fixed (SteerNet) / gate (SteerNet+Gate)")
print("=" * 80)

process = subprocess.Popen(
    _bench_cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")
if process.wait() != 0:
    print(f"[warning] Benchmark exited with code {process.returncode}")

print("\n" + "=" * 80)
print("3-Arm Benchmark Complete (V18)")
print("=" * 80)


## 11. Deep Diagnostic Report + Gate Behavior Analysis

In [ ]:
import os, json, re, collections, math
from pathlib import Path
import numpy as np
from src.evaluation.chain import audit_chain, gold_step_values, gold_step_recall, classify_outcome

SEP = '=' * 80
THIN = '-' * 60

def _gold_val(gold_text):
    """Extract numeric gold answer — handles both '#### 18' and plain '18' formats."""
    s = str(gold_text).strip()
    if '####' in s:
        s = s.split('####')[-1].strip()
    try:
        return float(s.replace(',', '').replace('$', ''))
    except (ValueError, TypeError):
        return None

# ═══════════════════════════════════════════════════════════════
# LOAD ALL DATA
# ═══════════════════════════════════════════════════════════════
training_log = None
if os.path.exists("report/training_log.json"):
    with open("report/training_log.json", encoding="utf-8") as f:
        training_log = json.load(f)

record = None
if os.path.exists("experiment_record.json"):
    with open("experiment_record.json", encoding="utf-8") as f:
        record = json.load(f)

gen_data = {}
for shot_str in (record or {}).get("shots", {}):
    gen_data[shot_str] = {}
    for arm in ['baseline', 'fixed']:
        path = f"report/generations/gsm8k_{shot_str}shot_{arm}.jsonl"
        if os.path.exists(path):
            with open(path, encoding='utf-8') as f:
                gen_data[shot_str][arm] = [json.loads(line) for line in f if line.strip()]
    paired_path = f"report/generations/gsm8k_{shot_str}shot.jsonl"
    if os.path.exists(paired_path):
        with open(paired_path, encoding='utf-8') as f:
            gen_data[shot_str]['paired'] = [json.loads(line) for line in f if line.strip()]

# ═══════════════════════════════════════════════════════════════
# 1. TRAINING CONVERGENCE
# ═══════════════════════════════════════════════════════════════
print(SEP)
print("  1. TRAINING CONVERGENCE")
print(SEP)
if training_log and training_log.get("epochs"):
    epochs = [ep for ep in training_log["epochs"] if isinstance(ep, dict)]
    print(f"  {'Epoch':<7} {'val_nll':>10} {'crit':>10} {'result_top1':>12}")
    print(f"  {THIN}")
    for i, ep in enumerate(epochs):
        print(f"  {i+1:<7} {ep.get('validation_nll',0):>10.4f} {ep.get('crit',0):>10.4f} {ep.get('result_top1',0):>12.3f}")

    val_nlls = [ep.get('validation_nll', 0) for ep in epochs]
    crits = [ep.get('crit', 0) for ep in epochs]
    if len(val_nlls) >= 2:
        total_drop = val_nlls[0] - val_nlls[-1]
        last_drop = val_nlls[-2] - val_nlls[-1]
        crit_drop = crits[0] - crits[-1]
        print(f"\n  val_nll: {val_nlls[0]:.4f} -> {val_nlls[-1]:.4f} (drop={total_drop:.4f})")
        print(f"  crit:    {crits[0]:.4f} -> {crits[-1]:.4f} (drop={crit_drop:.4f})")
        if val_nlls[-1] > val_nlls[-2]:
            print("  VERDICT: DIVERGING — val_nll increased in last epoch!")
        elif last_drop < 0.002:
            print("  VERDICT: CONVERGED (plateau reached)")
        elif last_drop > 0.01:
            print("  VERDICT: STILL IMPROVING — consider more epochs")
        else:
            print("  VERDICT: CONVERGING (diminishing returns)")
else:
    print("  No training log found.")

# ═══════════════════════════════════════════════════════════════
# 2. ACCURACY TABLE
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  2. ACCURACY RESULTS")
print(SEP)
if record:
    shots_data = record.get("shots", {})
    for shot_str, shot_data in shots_data.items():
        b = shot_data.get("baseline", {})
        s = shot_data.get("fixed", {})
        paired = shot_data.get("paired", {})

        print(f"\n  [{shot_str}-shot] (n={b.get('n', '?')})")
        print(f"  {'':>18} {'Baseline':>10} {'Fixed':>10} {'Delta':>10}")
        print(f"  {THIN}")

        b_strict = b.get('strict_accuracy', 0)
        s_strict = s.get('strict_accuracy', 0)
        b_len = b.get('lenient_accuracy', 0)
        s_len = s.get('lenient_accuracy', 0)
        b_fmt = b.get('format_rate', 0) * 100
        s_fmt = s.get('format_rate', 0) * 100

        print(f"  {'Strict accuracy':<18} {b_strict:>9.1f}% {s_strict:>9.1f}% {s_strict-b_strict:>+9.1f}pp")
        print(f"  {'Lenient accuracy':<18} {b_len:>9.1f}% {s_len:>9.1f}% {s_len-b_len:>+9.1f}pp")
        print(f"  {'Format rate':<18} {b_fmt:>9.1f}% {s_fmt:>9.1f}% {s_fmt-b_fmt:>+9.1f}pp")

        w = paired.get('fixed_wins', 0)
        l = paired.get('fixed_losses', 0)
        t = paired.get('ties', 0)
        ci = paired.get('ci95_pp', [0, 0])
        p = paired.get('mcnemar_p_value', 1.0)
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f"\n  Paired: {w}W / {l}L / {t}T  |  delta={paired.get('delta_pp',0):+.1f}pp  CI=[{ci[0]:+.1f},{ci[1]:+.1f}]  p={p:.4f} ({sig})")

        guard = s.get('guard_trip_rate', 0)
        ident = paired.get('identical_text_rate', 0)
        print(f"  Guard trips: {guard:.3f}  |  Identical text: {ident:.1%}")

        if s_strict - b_strict > 5 and s_len - b_len < -2:
            print(f"  >> FORMAT-ONLY: strict up but lenient DOWN — learned format, hurt reasoning")
        elif s_len - b_len > 3:
            print(f"  >> REASONING IMPROVED: lenient accuracy gained")
        elif s_len - b_len < -3:
            print(f"  >> REASONING DEGRADED: lenient accuracy dropped")
        elif ident > 0.9:
            print(f"  >> NULL INTERVENTION: >90% identical text — steering has no effect")

# ═══════════════════════════════════════════════════════════════
# 3. STEERING MAGNITUDE (rel_max)
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  3. STEERING MAGNITUDE (rel_max = ||delta|| / ||hidden||)")
print(SEP)
for shot_str in gen_data:
    fixed = gen_data[shot_str].get('fixed', [])
    baseline = gen_data[shot_str].get('baseline', [])
    if not fixed:
        continue
    rel_maxes = np.array([r.get('rel_max', 0) for r in fixed])
    print(f"\n  {shot_str}-shot fixed ({len(rel_maxes)} questions):")
    print(f"    mean={rel_maxes.mean():.4f}  median={np.median(rel_maxes):.4f}  "
          f"p95={np.percentile(rel_maxes,95):.4f}  max={rel_maxes.max():.4f}")

    std = rel_maxes.std()
    print(f"    std={std:.4f}  min={rel_maxes.min():.4f}  range={rel_maxes.max()-rel_maxes.min():.4f}")

    if rel_maxes.max() > 0.50:
        print(f"    CRITICAL: deltas are {rel_maxes.max():.0%} of hidden norm — overriding model!")
    elif rel_maxes.max() > 0.15:
        print(f"    HIGH: deltas up to {rel_maxes.max():.0%} of hidden norm — may disrupt reasoning")
    elif rel_maxes.max() < 0.005:
        print(f"    TOO WEAK: max rel_max={rel_maxes.max():.4f} — steering negligible")
    else:
        print(f"    OK: deltas are {rel_maxes.max():.1%} of hidden norm — gentle nudge")

    if std < 0.002:
        print(f"    WARNING: std={std:.4f} — mag_head likely SATURATED (same magnitude for all questions)")
    elif std > 0.01:
        print(f"    GOOD: std={std:.4f} — per-question magnitude variation detected")

    # Distribution buckets
    buckets = [(0, 0.005), (0.005, 0.01), (0.01, 0.02), (0.02, 0.03), (0.03, 0.05), (0.05, 0.10), (0.10, 0.50), (0.50, 2.0)]
    for lo, hi in buckets:
        count = ((rel_maxes >= lo) & (rel_maxes < hi)).sum()
        if count > 0:
            print(f"    [{lo:.3f}, {hi:.3f}): {count} questions ({count/len(rel_maxes):.0%})")

# ═══════════════════════════════════════════════════════════════
# 4. REASONING QUALITY (EAA, CAR, GSR)
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  4. REASONING QUALITY")
print(SEP)
print("  EAA = Exact Arithmetic Accuracy (% of equations computed correctly)")
print("  CAR = Clean Arithmetic Rate (% of responses with ALL equations correct)")
print("  GSR = Gold Step Recall (% of gold intermediate values found in output)")

for shot_str in gen_data:
    paired = gen_data[shot_str].get('paired', [])
    if not paired:
        continue

    metrics = {'b_eaa': [], 'f_eaa': [], 'b_car': [], 'f_car': [],
               'b_gsr': [], 'f_gsr': [], 'b_neq': [], 'f_neq': []}

    for row in paired:
        b_text = row.get('baseline', {}).get('text', '')
        f_text = row.get('fixed', {}).get('text', '')
        question = row.get('question', '')
        gold_text = row.get('gold', '')

        b_audit = audit_chain(b_text, question)
        f_audit = audit_chain(f_text, question)

        metrics['b_eaa'].append(b_audit['eaa'])
        metrics['f_eaa'].append(f_audit['eaa'])
        metrics['b_car'].append(b_audit['car'])
        metrics['f_car'].append(f_audit['car'])
        metrics['b_neq'].append(b_audit['total_equations'])
        metrics['f_neq'].append(f_audit['total_equations'])

        gv = gold_step_values(gold_text)
        metrics['b_gsr'].append(gold_step_recall(b_text, gv))
        metrics['f_gsr'].append(gold_step_recall(f_text, gv))

    print(f"\n  {shot_str}-shot ({len(paired)} questions):")
    print(f"  {'Metric':<28} {'Baseline':>10} {'Fixed':>10} {'Delta':>10}")
    print(f"  {THIN}")
    for label, bk, fk in [
        ('EAA (equation accuracy)', 'b_eaa', 'f_eaa'),
        ('CAR (clean chain rate)', 'b_car', 'f_car'),
        ('GSR (gold step recall)', 'b_gsr', 'f_gsr'),
        ('Avg equations/response', 'b_neq', 'f_neq'),
    ]:
        bv = np.mean(metrics[bk])
        fv = np.mean(metrics[fk])
        d = fv - bv
        if 'neq' in bk:
            print(f"  {label:<28} {bv:>10.1f} {fv:>10.1f} {d:>+10.1f}")
        else:
            print(f"  {label:<28} {bv:>9.1%} {fv:>9.1%} {d:>+9.1%}")

    eaa_d = np.mean(metrics['f_eaa']) - np.mean(metrics['b_eaa'])
    car_d = np.mean(metrics['f_car']) - np.mean(metrics['b_car'])
    gsr_d = np.mean(metrics['f_gsr']) - np.mean(metrics['b_gsr'])

    if eaa_d < -0.03:
        print(f"  >> ARITHMETIC DEGRADED: model makes more calculation errors ({eaa_d:+.1%})")
    if car_d < -0.03:
        print(f"  >> CHAIN CORRUPTED: fewer fully-correct reasoning chains ({car_d:+.1%})")
    if gsr_d < -0.05:
        print(f"  >> REASONING DRIFT: missing gold intermediate values ({gsr_d:+.1%})")
    if eaa_d >= 0 and car_d >= 0 and gsr_d >= 0:
        print(f"  >> Reasoning quality maintained or improved")

# ═══════════════════════════════════════════════════════════════
# 5. ERROR TAXONOMY (per arm)
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  5. ERROR TAXONOMY")
print(SEP)
print("  Categories: CORRECT | PLAN_ERROR (wrong approach) | ARITHMETIC (bad calc)")
print("  UNGROUNDED (hallucinated number) | EXTRACTION (right intermediate, wrong final)")
print("  NO_ANSWER | LOOP_OR_RUNON | TRUNC_NUMBER")

for shot_str in gen_data:
    paired = gen_data[shot_str].get('paired', [])
    if not paired:
        continue

    b_tax = collections.Counter()
    f_tax = collections.Counter()

    for row in paired:
        b_res = row.get('baseline', {})
        f_res = row.get('fixed', {})
        question = row.get('question', '')
        gv = _gold_val(row.get('gold', ''))
        if gv is None:
            continue

        b_cls = classify_outcome(b_res, gv, question=question)
        f_cls = classify_outcome(f_res, gv, question=question)
        b_tax[b_cls] += 1
        f_tax[f_cls] += 1

    total = sum(b_tax.values())
    print(f"\n  {shot_str}-shot ({total} questions):")
    print(f"  {'Category':<22} {'Baseline':>10} {'Fixed':>10} {'Delta':>10} {'Direction'}")
    print(f"  {THIN}")
    all_cats = sorted(set(list(b_tax.keys()) + list(f_tax.keys())),
                      key=lambda x: -(f_tax[x] + b_tax[x]))
    for cat in all_cats:
        bv = b_tax[cat]
        fv = f_tax[cat]
        d = fv - bv
        arrow = 'BETTER' if (cat == 'CORRECT' and d > 0) or (cat != 'CORRECT' and d < 0) else (
                'WORSE' if (cat == 'CORRECT' and d < 0) or (cat != 'CORRECT' and d > 0) else '=')
        print(f"  {cat:<22} {bv:>10} {fv:>10} {d:>+10} {arrow}")

# ═══════════════════════════════════════════════════════════════
# 6. FLIP ANALYSIS (Win/Loss examples)
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  6. FLIP ANALYSIS (questions that changed outcome)")
print(SEP)

for shot_str in gen_data:
    paired = gen_data[shot_str].get('paired', [])
    if not paired:
        continue

    wins, losses = [], []
    for row in paired:
        b_ok = row.get('baseline', {}).get('lenient', False)
        f_ok = row.get('fixed', {}).get('lenient', False)
        if f_ok and not b_ok:
            wins.append(row)
        elif b_ok and not f_ok:
            losses.append(row)

    print(f"\n  {shot_str}-shot: {len(wins)} WINS, {len(losses)} LOSSES")

    if wins:
        print(f"\n  -- WINS (baseline wrong -> fixed right) --")
        win_reasons = collections.Counter()
        for w in wins:
            gv = _gold_val(w.get('gold', ''))
            if gv is None:
                continue
            cls = classify_outcome(w['baseline'], gv, question=w.get('question', ''))
            win_reasons[f"fixed {cls}"] += 1
        for reason, count in win_reasons.most_common(5):
            print(f"    {count:>3}x baseline was {reason}")
        for w in wins[:3]:
            ga = _gold_val(w.get('gold', ''))
            bp = w['baseline'].get('pred_lenient', '?')
            fp = w['fixed'].get('pred_lenient', '?')
            print(f"\n    Q: {w.get('question','')[:90]}...")
            print(f"    Gold={ga}  Baseline={bp}  Fixed={fp}")

    if losses:
        print(f"\n  -- LOSSES (baseline right -> fixed wrong) --")
        loss_reasons = collections.Counter()
        for lo in losses:
            gv = _gold_val(lo.get('gold', ''))
            if gv is None:
                continue
            cls = classify_outcome(lo['fixed'], gv, question=lo.get('question', ''))
            loss_reasons[cls] += 1
        for reason, count in loss_reasons.most_common(5):
            print(f"    {count:>3}x fixed error: {reason}")
        for lo in losses[:3]:
            ga = _gold_val(lo.get('gold', ''))
            bp = lo['baseline'].get('pred_lenient', '?')
            fp = lo['fixed'].get('pred_lenient', '?')
            print(f"\n    Q: {lo.get('question','')[:90]}...")
            print(f"    Gold={ga}  Baseline={bp}  Fixed={fp}")
            ft = lo['fixed'].get('text', '')
            if ft:
                lines = ft.strip().split('\n')
                print(f"    Fixed output ({len(lines)} lines):")
                for line in lines[:5]:
                    print(f"      {line}")
                if len(lines) > 5:
                    print(f"      ... ({len(lines)-5} more lines)")

# ═══════════════════════════════════════════════════════════════
# 7. GENERATION STATS
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  7. GENERATION STATS")
print(SEP)
for shot_str in gen_data:
    for arm in ['baseline', 'fixed']:
        data = gen_data[shot_str].get(arm, [])
        if not data:
            continue
        stops = collections.Counter(r.get('stop_reason', '?') for r in data)
        tokens = [r.get('n_new_tokens', 0) for r in data]
        print(f"  {shot_str}-shot {arm}: tokens mean={np.mean(tokens):.0f} med={np.median(tokens):.0f} max={max(tokens)}  |  stops: {dict(stops)}")

# ═══════════════════════════════════════════════════════════════
# 8. DIFFICULTY ANALYSIS (easy vs hard questions)
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  8. DIFFICULTY STRATIFICATION")
print(SEP)
print("  Easy = baseline gets right, Hard = baseline gets wrong")

for shot_str in gen_data:
    paired = gen_data[shot_str].get('paired', [])
    if not paired:
        continue

    easy_b, easy_f, hard_b, hard_f = 0, 0, 0, 0
    n_easy, n_hard = 0, 0
    for row in paired:
        b_ok = row.get('baseline', {}).get('lenient', False)
        f_ok = row.get('fixed', {}).get('lenient', False)
        if b_ok:
            n_easy += 1
            easy_b += 1
            easy_f += int(f_ok)
        else:
            n_hard += 1
            hard_f += int(f_ok)

    if n_easy > 0 and n_hard > 0:
        print(f"\n  {shot_str}-shot:")
        print(f"    Easy ({n_easy}): baseline={easy_b/n_easy:.0%} -> fixed={easy_f/n_easy:.0%}  "
              f"(retention={easy_f/n_easy:.0%}, lost={n_easy-easy_f})")
        print(f"    Hard ({n_hard}): baseline=0% -> fixed={hard_f/n_hard:.0%}  "
              f"(gained={hard_f})")
        net = hard_f - (n_easy - easy_f)
        print(f"    Net: +{hard_f} gained on hard, -{n_easy-easy_f} lost on easy = {net:+d} net flips")
        if n_easy - easy_f > hard_f:
            print(f"    >> PROBLEM: losing more easy questions than gaining hard ones!")
        elif hard_f > 0 and n_easy == easy_f:
            print(f"    >> EXCELLENT: gaining on hard without losing any easy ones!")

# ═══════════════════════════════════════════════════════════════
# 9. AUTO-DIAGNOSIS SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  9. AUTO-DIAGNOSIS")
print(SEP)
signals = []
if record:
    for shot_str, shot_data in record.get("shots", {}).items():
        b = shot_data.get("baseline", {})
        s = shot_data.get("fixed", {})
        paired = shot_data.get("paired", {})

        len_d = s.get('lenient_accuracy', 0) - b.get('lenient_accuracy', 0)
        str_d = s.get('strict_accuracy', 0) - b.get('strict_accuracy', 0)
        fmt_d = (s.get('format_rate', 0) - b.get('format_rate', 0)) * 100
        ident = paired.get('identical_text_rate', 0)

        fixed_gen = gen_data.get(shot_str, {}).get('fixed', [])
        rel_maxes = np.array([r.get('rel_max', 0) for r in fixed_gen]) if fixed_gen else np.array([0])
        max_rm = rel_maxes.max()
        std_rm = rel_maxes.std()

        if ident > 0.90:
            signals.append(f"  [{shot_str}s] NULL INTERVENTION: {ident:.0%} identical text — steering has NO effect")
            if max_rm < 0.005:
                signals.append(f"         Cause: rel_max={max_rm:.4f} — deltas too small to change output")
        elif str_d > 10 and len_d < -2:
            signals.append(f"  [{shot_str}s] FORMAT-ONLY: strict +{str_d:.0f}pp but lenient {len_d:+.0f}pp")
            signals.append(f"         Cause: learned 'Final answer:' format but corrupted reasoning")
            if max_rm > 0.3:
                signals.append(f"         rel_max={max_rm:.2f} — deltas too large, overriding base model")
        elif len_d < -5:
            signals.append(f"  [{shot_str}s] SIGNIFICANT REGRESSION: lenient {len_d:+.0f}pp")
            if max_rm > 0.3:
                signals.append(f"         Likely cause: over-steering (rel_max={max_rm:.2f})")
            else:
                signals.append(f"         rel_max={max_rm:.3f} — check error taxonomy for root cause")
        elif len_d > 5:
            signals.append(f"  [{shot_str}s] GOOD IMPROVEMENT: lenient {len_d:+.0f}pp (p={paired.get('mcnemar_p_value',1):.4f})")
        elif abs(len_d) <= 2:
            signals.append(f"  [{shot_str}s] NEUTRAL: lenient {len_d:+.1f}pp — within noise")
            if max_rm < 0.01:
                signals.append(f"         Possible cause: under-steering (rel_max={max_rm:.4f})")
            if std_rm < 0.002 and max_rm > 0.01:
                signals.append(f"         Possible cause: mag_head saturated (std={std_rm:.4f}) — no per-question selectivity")
        else:
            signals.append(f"  [{shot_str}s] MILD {'GAIN' if len_d > 0 else 'DROP'}: lenient {len_d:+.1f}pp")

        if std_rm < 0.002 and max_rm > 0.01:
            signals.append(f"  [{shot_str}s] MAG_HEAD SATURATED: std={std_rm:.4f}, all questions get same magnitude")

for s in signals:
    print(s)

if not signals:
    print("  No data available for diagnosis.")

print(f"\n{SEP}")
print("  END OF DIAGNOSTIC REPORT")
print(SEP)

# ═══════════════════════════════════════════════════════════════
# TRAINING CURVES PLOT (saved to report/)
# ═══════════════════════════════════════════════════════════════
if training_log:
    from src.evaluation.diagnostics import plot_training
    plot_training(training_log, "report/training_curves.png")
    print("\nTraining curves saved to report/training_curves.png")

## 11a. 3-Arm Comparison: Baseline vs SteerNet-Only vs SteerNet+Gate

Reads the single experiment_record.json with all 3 arms.
Shows delta from baseline for both the fixed (SteerNet) and gate (SteerNet+Gate) arms.

In [ ]:
import os, json
SEP = "=" * 80
THIN = "-" * 60

rec = None
if os.path.exists("experiment_record.json"):
    with open("experiment_record.json") as f:
        rec = json.load(f)

if rec:
    print(SEP)
    print("  3-ARM COMPARISON: Baseline vs SteerNet vs SteerNet+Gate")
    print(SEP)

    all_shots = sorted(rec.get("shots", {}).keys())
    for shot_str in all_shots:
        data = rec["shots"][shot_str]
        b = data.get("baseline", {})
        f = data.get("fixed", {})
        g = data.get("gate", {})

        if not b:
            continue

        b_strict = b.get("strict_accuracy", 0)
        b_len    = b.get("lenient_accuracy", 0)
        b_fmt    = b.get("format_rate", 0) * 100

        f_strict_d = f.get("strict_accuracy", 0) - b_strict if f else 0
        f_len_d    = f.get("lenient_accuracy", 0) - b_len if f else 0
        f_fmt_d    = (f.get("format_rate", 0) * 100 - b_fmt) if f else 0

        g_strict_d = g.get("strict_accuracy", 0) - b_strict if g else 0
        g_len_d    = g.get("lenient_accuracy", 0) - b_len if g else 0
        g_fmt_d    = (g.get("format_rate", 0) * 100 - b_fmt) if g else 0

        hdr_metric = "Metric"
        hdr_fixed  = "Fixed (SN)"
        hdr_gate   = "Gate (SN+G)"
        hdr_diff   = "Gate - Fixed"
        lbl_strict = "Strict delta (pp)"
        lbl_lenient = "Lenient delta (pp)"
        lbl_format = "Format delta (pp)"

        print(f"\n  [{shot_str}-shot]  Baseline: strict={b_strict:.1f}  lenient={b_len:.1f}  format={b_fmt:.0f}%")
        print(f"  {hdr_metric:<22} {hdr_fixed:>12} {hdr_gate:>12} {hdr_diff:>12}")
        print(f"  {THIN}")
        print(f"  {lbl_strict:<22} {f_strict_d:>+11.1f}  {g_strict_d:>+11.1f}  {g_strict_d - f_strict_d:>+11.1f}")
        print(f"  {lbl_lenient:<22} {f_len_d:>+11.1f}  {g_len_d:>+11.1f}  {g_len_d - f_len_d:>+11.1f}")
        print(f"  {lbl_format:<22} {f_fmt_d:>+11.1f}  {g_fmt_d:>+11.1f}  {g_fmt_d - f_fmt_d:>+11.1f}")

        gate_vs_fixed = g_len_d - f_len_d
        if gate_vs_fixed > 2:
            print(f"  >> GATE HELPS: +{gate_vs_fixed:.1f}pp over SteerNet-only")
        elif gate_vs_fixed < -2:
            print(f"  >> GATE HURTS: {gate_vs_fixed:.1f}pp worse than SteerNet-only")
        else:
            print(f"  >> GATE NEUTRAL: {gate_vs_fixed:+.1f}pp vs SteerNet-only")

    print(f"\n{SEP}")
    print("  END OF 3-ARM COMPARISON")
    print(SEP)
else:
    print("No experiment_record.json found")


## 11b. Gate Behavior Deep Analysis (V18)

Runs inference on validation data with gate active. Reports:
- **Gate training convergence** (from checkpoint)
- **Gate weight analysis** (per-layer parameter statistics)
- **Per-token-type alpha** (math/format/prose differentiation)
- **Per-layer alpha** (layer-specific gating behavior)
- **Gate selectivity score** (variance = discrimination ability)

In [ ]:
import os, json, gc, torch
import numpy as np
from collections import defaultdict

SEP = '=' * 80
THIN = '-' * 60
GATE_CKPT = "checkpoints/gate.pt"
SN_GATED  = "checkpoints/steernet_gated.pt"

if not RUN_GATE or not os.path.exists(GATE_CKPT):
    print("Gate not trained — skipping Gate analysis")
else:
    # ═══════════════════════════════════════════════════════════════
    # A. GATE TRAINING CONVERGENCE
    # ═══════════════════════════════════════════════════════════════
    print(SEP)
    print("  A. GATE TRAINING CONVERGENCE")
    print(SEP)

    gate_ckpt = torch.load(GATE_CKPT, map_location="cpu", weights_only=False)
    history = gate_ckpt.get("history", {})
    gate_epochs_list = history.get("epochs", [])

    if gate_epochs_list:
        print(f"  {'Epoch':<8} {'train_loss':>12} {'val_nll':>12}")
        print(f"  {THIN}")
        for ep in gate_epochs_list:
            print(f"  {ep['epoch']:<8} {ep['train_loss']:>12.4f} {ep['val_nll']:>12.4f}")
        if len(gate_epochs_list) >= 2:
            vn = [e['val_nll'] for e in gate_epochs_list]
            drop = vn[0] - vn[-1]
            print(f"\n  val_nll: {vn[0]:.4f} -> {vn[-1]:.4f} (drop={drop:.4f})")
            if drop > 0.01:
                print("  Gate is learning — val_nll improving")
            elif drop > 0:
                print("  Gate learning slowly — consider more epochs")
            else:
                print("  WARNING: Gate val_nll not improving")
    else:
        gate_log_path = "report/gate_training_log.json"
        if os.path.exists(gate_log_path):
            with open(gate_log_path) as f:
                history = json.load(f)
            for ep in history.get("epochs", []):
                print(f"  Epoch {ep['epoch']}: train_loss={ep['train_loss']:.4f}  val_nll={ep['val_nll']:.4f}")

    print(f"\n  Best val_nll: {gate_ckpt.get('best_val_nll', 'N/A')}")
    print(f"  Gate layers: {gate_ckpt.get('fixed_layers')}")
    print(f"  Gate epochs: {gate_ckpt.get('gate_epochs')}")

    # ═══════════════════════════════════════════════════════════════
    # B. GATE WEIGHT ANALYSIS
    # ═══════════════════════════════════════════════════════════════
    print(f"\n{SEP}")
    print("  B. GATE WEIGHT ANALYSIS (per-layer parameter stats)")
    print(SEP)

    gate_state = gate_ckpt.get("gates", {})
    layer_keys = sorted(set(k.split('.')[0] for k in gate_state.keys()))

    for layer_key in layer_keys:
        print(f"\n  Layer {layer_key}:")
        for param_name in ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']:
            full_key = f"{layer_key}.{param_name}"
            if full_key in gate_state:
                t = gate_state[full_key].float()
                shape_str = str(list(t.shape))
                if t.numel() > 1:
                    print(f"    {param_name:<14} shape={shape_str:<14s} "
                          f"mean={t.mean():.4f}  std={t.std():.4f}  "
                          f"min={t.min():.4f}  max={t.max():.4f}")
                else:
                    print(f"    {param_name:<14} shape={shape_str:<14s} "
                          f"value={t.item():.4f}")

        fc2_bias_key = f"{layer_key}.fc2.bias"
        if fc2_bias_key in gate_state:
            logit = float(gate_state[fc2_bias_key])
            alpha_init = 1.0 / (1.0 + np.exp(-logit))
            print(f"    fc2.bias logit={logit:.4f} -> sigmoid={alpha_init:.4f} (init was ~0.80)")
            if alpha_init > 0.9:
                print(f"    >> Gate nearly always ON (alpha~{alpha_init:.2f}) — may not be selective")
            elif alpha_init < 0.3:
                print(f"    >> Gate nearly always OFF (alpha~{alpha_init:.2f}) — steering suppressed")
            else:
                print(f"    >> Gate in active range ({alpha_init:.2f}) — good selectivity potential")

    # ═══════════════════════════════════════════════════════════════
    # C. GATE ALPHA INFERENCE ANALYSIS
    # ═══════════════════════════════════════════════════════════════
    print(f"\n{SEP}")
    print("  C. GATE ALPHA INFERENCE (per-token-type & per-layer)")
    print(SEP)
    print("  Running forward passes on validation data to collect gate alpha values...")

    from src.configs.config import RSCConfig, apply_overrides
    from src.models.wrapper import Phi2WithRSC
    from src.gating.gate import RSCUsefulnessGate
    from src.training.trainer import _prepare
    from src.training.losses import T_FORMAT, T_OPERAND, T_OPERATOR, T_PROSE, T_RESULT, T_FINAL
    from src.data.dataset import gsm8k_to_cot
    from scripts.run_ars_pipeline import prepare_dataset_split
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    cfg = RSCConfig()
    overrides = [
        f"fixed_layers={S_FIXED}", f"candidate_layers={S_FIXED}",
        f"lora_rank={LORA_RANK}", f"lora_alpha={LORA_ALPHA}",
        f"amplitude_mode={AMPLITUDE_MODE}", f"rho_max={RHO_MAX}",
        f"steer_kappa={STEER_KAPPA}", f"mag_head_init_bias={MAG_HEAD_INIT}",
        f"loss_mode={LOSS_MODE}", f"prompt_shielding=false",
        f"max_tok_len={MAX_TOK_LEN}", f"use_gate=true",
    ]
    cfg = apply_overrides(cfg, overrides)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    backbone = AutoModelForCausalLM.from_pretrained(
        cfg.model_name, torch_dtype=torch.float16, trust_remote_code=True
    )
    model = Phi2WithRSC(backbone, cfg, device).to(device).eval()

    from src.training.trainer import load_steer_weights
    sn_path = SN_GATED if os.path.exists(SN_GATED) else "checkpoints/steernet.pt"
    load_steer_weights(model, sn_path, cfg, device)

    import torch.nn as nn_mod
    hidden_size = model.base_model.config.hidden_size
    model.gates = nn_mod.ModuleDict({
        str(layer): RSCUsefulnessGate(hidden_size, cfg).to(device)
        for layer in model.target_layers
    })
    from src.training.gate_trainer import load_gate_weights
    load_gate_weights(model, GATE_CKPT, device)

    target_layers = model.target_layers
    gates_ref = model.gates
    def _gate_provider(position, hidden, delta):
        return gates_ref[str(target_layers[position])](hidden, delta)
    model.set_gate_provider(_gate_provider)
    model.set_routing_override("only", list(target_layers))

    # Load validation data via the same pipeline the trainer uses
    raw_train, raw_val = gsm8k_to_cot(tokenizer, cfg)
    val_rows = prepare_dataset_split(raw_val, limit=100)

    TOKEN_TYPE_NAMES = {T_FORMAT: 'FORMAT', T_OPERATOR: 'OPERATOR', T_OPERAND: 'OPERAND',
                        T_RESULT: 'RESULT', T_FINAL: 'FINAL', T_PROSE: 'PROSE'}

    per_layer_alphas = defaultdict(list)
    per_type_alphas = defaultdict(lambda: defaultdict(list))
    all_alphas = []

    with torch.no_grad():
        for start in range(0, len(val_rows), 2):
            batch_rows = val_rows[start:start+2]
            if not batch_rows:
                continue
            batch, extras = _prepare(batch_rows, device)
            ids = batch["input_ids"]
            mask = batch["attention_mask"]
            labels = batch.get("labels")
            token_types = extras.get("token_types")

            model(input_ids=ids, attention_mask=mask)

            valid = mask[:, :ids.shape[1]].bool()
            if labels is not None:
                valid = valid & labels[:, :ids.shape[1]].ne(-100)

            for layer_key, gate in model.gates.items():
                if gate.last_alpha is None:
                    continue
                alpha = gate.last_alpha[:, :valid.shape[1], 0]
                for b in range(alpha.shape[0]):
                    v = valid[b]
                    a = alpha[b][v].cpu().numpy()
                    per_layer_alphas[layer_key].extend(a.tolist())
                    all_alphas.extend(a.tolist())

                    if token_types is not None:
                        tt = token_types[b, :valid.shape[1]]
                        for tok_type_id, tok_name in TOKEN_TYPE_NAMES.items():
                            type_mask = v & (tt == tok_type_id)
                            if type_mask.any():
                                type_a = alpha[b][type_mask].cpu().numpy()
                                per_type_alphas[layer_key][tok_name].extend(type_a.tolist())

    all_alphas = np.array(all_alphas)
    print(f"\n  Collected {len(all_alphas)} alpha values from {len(val_rows)} validation samples")

    # ── Overall alpha distribution ──
    print(f"\n  Overall alpha: mean={all_alphas.mean():.4f}  std={all_alphas.std():.4f}  "
          f"min={all_alphas.min():.4f}  max={all_alphas.max():.4f}  median={np.median(all_alphas):.4f}")

    buckets = [(0, 0.1), (0.1, 0.3), (0.3, 0.5), (0.5, 0.7), (0.7, 0.9), (0.9, 1.01)]
    print("  Alpha distribution:")
    for lo, hi in buckets:
        count = ((all_alphas >= lo) & (all_alphas < hi)).sum()
        pct = count / len(all_alphas)
        bar = '#' * int(pct * 40)
        print(f"    [{lo:.1f}, {hi:.1f}): {count:>6} ({pct:>5.1%}) {bar}")

    # ── Per-layer breakdown ──
    print(f"\n  Per-layer alpha statistics:")
    print(f"  {'Layer':<10} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Median':>8}")
    print(f"  {THIN}")
    for layer_key in sorted(per_layer_alphas.keys(), key=int):
        a = np.array(per_layer_alphas[layer_key])
        print(f"  {layer_key:<10} {a.mean():>8.4f} {a.std():>8.4f} {a.min():>8.4f} {a.max():>8.4f} {np.median(a):>8.4f}")

    # ── Per-token-type breakdown (aggregated across layers) ──
    print(f"\n  Per-token-type alpha (aggregated across all layers):")
    print(f"  {'Type':<12} {'Mean':>8} {'Std':>8} {'Count':>8} {'Verdict'}")
    print(f"  {THIN}")

    type_means = {}
    all_type_alphas = defaultdict(list)
    for layer_key in per_type_alphas:
        for tok_name, vals in per_type_alphas[layer_key].items():
            all_type_alphas[tok_name].extend(vals)

    target_alphas = {'OPERATOR': 0.85, 'OPERAND': 0.85, 'RESULT': 0.85, 'FINAL': 0.85,
                     'FORMAT': 0.15, 'PROSE': 0.65}

    for tok_name in ['OPERATOR', 'OPERAND', 'RESULT', 'FINAL', 'FORMAT', 'PROSE']:
        vals = all_type_alphas.get(tok_name, [])
        if not vals:
            print(f"  {tok_name:<12} {'N/A':>8} {'N/A':>8} {0:>8}")
            continue
        a = np.array(vals)
        target = target_alphas.get(tok_name, 0.5)
        diff = abs(a.mean() - target)
        verdict = 'ON TARGET' if diff < 0.15 else ('HIGH' if a.mean() > target else 'LOW')
        type_means[tok_name] = a.mean()
        print(f"  {tok_name:<12} {a.mean():>8.4f} {a.std():>8.4f} {len(vals):>8} {verdict} (target={target:.2f})")

    # ── Per-layer x per-type matrix ──
    print(f"\n  Per-layer x per-type alpha matrix:")
    types_ordered = ['OPERATOR', 'OPERAND', 'RESULT', 'FINAL', 'FORMAT', 'PROSE']
    header = f"  {'Layer':<8}" + "".join(f"{t:>10}" for t in types_ordered)
    print(header)
    print(f"  {THIN}")
    for layer_key in sorted(per_type_alphas.keys(), key=int):
        row = f"  {layer_key:<8}"
        for tok_name in types_ordered:
            vals = per_type_alphas[layer_key].get(tok_name, [])
            if vals:
                row += f"{np.mean(vals):>10.4f}"
            else:
                row += f"{'---':>10}"
        print(row)

    # ── Gate selectivity score ──
    print(f"\n  Gate Selectivity Metrics:")
    if type_means.get('RESULT') and type_means.get('FORMAT'):
        math_alpha = np.mean([type_means.get(t, 0) for t in ['OPERATOR', 'OPERAND', 'RESULT', 'FINAL'] if t in type_means])
        format_alpha = type_means.get('FORMAT', 0)
        prose_alpha = type_means.get('PROSE', 0)
        selectivity = math_alpha - format_alpha
        print(f"    Math avg alpha:   {math_alpha:.4f}")
        print(f"    Format alpha:     {format_alpha:.4f}")
        print(f"    Prose alpha:      {prose_alpha:.4f}")
        print(f"    Selectivity (math - format): {selectivity:.4f}")
        if selectivity > 0.3:
            print(f"    >> EXCELLENT: Gate strongly differentiates math ({math_alpha:.2f}) vs format ({format_alpha:.2f})")
        elif selectivity > 0.1:
            print(f"    >> GOOD: Gate shows some differentiation")
        elif selectivity > 0:
            print(f"    >> WEAK: Gate barely differentiates — may need more training")
        else:
            print(f"    >> FAILED: Gate does NOT differentiate math from format")

    overall_std = all_alphas.std()
    print(f"    Alpha variance (std): {overall_std:.4f}")
    if overall_std > 0.15:
        print(f"    >> HIGH VARIANCE: Gate is actively selecting which tokens to steer")
    elif overall_std > 0.05:
        print(f"    >> MODERATE: Some selectivity, but gate is relatively uniform")
    else:
        print(f"    >> LOW VARIANCE: Gate outputs nearly constant alpha — not selective")

    # ── Cleanup ──
    del model, backbone, tokenizer, raw_train, raw_val, val_rows
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"\n{SEP}")
    print("  END OF GATE ANALYSIS")
    print(SEP)

## 12. Save Artifacts


In [ ]:
import shutil, os, json, glob

os.makedirs("/kaggle/working/ars_artifacts", exist_ok=True)

for fname in [
    "deployment_contract.json",
    "experiment_record.json",
    "norms_calibration.json",
    "checkpoints/steernet.pt",
    "checkpoints/router.pt",
    "checkpoints/gate.pt",
    "checkpoints/steernet_gated.pt",
]:
    if os.path.exists(fname):
        shutil.copy(fname, "/kaggle/working/ars_artifacts/")
        print(f"Copied: {fname}")

if os.path.exists("report"):
    shutil.copytree("report", "/kaggle/working/ars_artifacts/report", dirs_exist_ok=True)
    print("Copied: report/")

for f in glob.glob("checkpoints/steernet_epoch*.pt"):
    shutil.copy(f, "/kaggle/working/ars_artifacts/")
    print(f"Copied: {f}")

if os.path.exists("report/gate_training_log.json"):
    shutil.copy("report/gate_training_log.json", "/kaggle/working/ars_artifacts/")
    print("Copied: report/gate_training_log.json")

print("\nAll artifacts saved to /kaggle/working/ars_artifacts/")


## 13. Export Benchmark Bundle (ZIP)

In [ ]:
import zipfile, os, shutil
from pathlib import Path

# Files needed to run any benchmark on another machine
BUNDLE_FILES = [
    "deployment_contract.json",   # route + envelope metadata
    "norms_calibration.json",     # per-layer norm caps (required by SteerNet)
    "checkpoints/steernet.pt",    # Phase 1 SteerNet weights
    "checkpoints/router.pt",      # Phase 2 Router weights (if trained)
]

ZIP_PATH = "/kaggle/working/ars_benchmark_bundle.zip"

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in BUNDLE_FILES:
        if os.path.exists(fpath):
            zf.write(fpath)
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"  + {fpath}  ({size_mb:.1f} MB)")
        else:
            print(f"  - {fpath}  [NOT FOUND — skipped]")

total_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"\n✅ Bundle saved: {ZIP_PATH}  ({total_mb:.1f} MB)")
print("Download this single file to run benchmarks on any machine.")